# Train EuroSAT Land Cover Classifier
Run this on Google Colab with free GPU (~30 min).
Then download the model files to your local machine.

In [ ]:
# Step 1: Mount Google Drive (to save model)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Install datasets library
!pip install datasets -q

In [ ]:
# Step 3: Load EuroSAT from Hugging Face
import tensorflow as tf
import numpy as np
from datasets import load_dataset

print('Loading EuroSAT RGB...')
hf = load_dataset('giswqs/EuroSAT_RGB')

def load(split):
    ds = hf[split]
    imgs, lbls = [], []
    for i in range(len(ds)):
        imgs.append(np.array(ds[i]['image'], dtype=np.float32) / 255.0)
        lbls.append(ds[i]['label'])
    return np.array(imgs), np.array(lbls)

x_train, y_train = load('train')
x_val, y_val = load('validation')
x_test, y_test = load('test')
print(f'Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}')

In [ ]:
# Step 4: Build MobileNetV2 model (lightweight, fast on GPU)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, RandomFlip, RandomBrightness, RandomContrast
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

base = MobileNetV2(include_top=False, weights='imagenet', input_shape=(64,64,3))
x = GlobalAveragePooling2D()(base.output)
x = Dense(512, activation='relu')(x)
x = Dropout(0.3)(x)
out = Dense(10, activation='softmax')(x)

model = Model(inputs=base.input, outputs=out)
for layer in model.layers:
    layer.trainable = True

model.compile(optimizer=Adam(0.0005),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
# Step 5: Data augmentation
aug = tf.keras.Sequential([
    RandomFlip('horizontal_and_vertical'),
    RandomBrightness(0.1),
    RandomContrast(0.1),
])

def make_ds(x, y, batch_size=128, shuffle=False, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(5000)
    if augment:
        ds = ds.map(lambda img, lbl: (aug(img, training=True), lbl),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

ds_train = make_ds(x_train, y_train, shuffle=True, augment=True)
ds_val = make_ds(x_val, y_val)
ds_test = make_ds(x_test, y_test)

In [ ]:
# Step 6: Train (target: ~30 min on Colab GPU)
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-6)
]

history = model.fit(ds_train, validation_data=ds_val,
                    epochs=20, callbacks=callbacks, verbose=1)

In [ ]:
# Step 7: Evaluate
test_loss, test_acc = model.evaluate(ds_test, verbose=1)
print(f'Test accuracy: {test_acc:.4f}')

In [ ]:
# Step 8: Save model + class indices
import os
os.makedirs('/content/drive/MyDrive/eurosat_model', exist_ok=True)

# Save as .h5 (compatible with the local app)
model.save('/content/drive/MyDrive/eurosat_model/ResNet50_eurosat.h5')

# Save class indices
CLASS_NAMES = {
    '0': 'AnnualCrop', '1': 'Forest', '2': 'HerbaceousVegetation',
    '3': 'Highway', '4': 'Industrial', '5': 'Pasture',
    '6': 'PermanentCrop', '7': 'Residential', '8': 'River', '9': 'SeaLake'
}
np.save('/content/drive/MyDrive/eurosat_model/class_indices.npy', CLASS_NAMES)

print('Model saved to Google Drive!')
print('Files:')
!ls -lh /content/drive/MyDrive/eurosat_model/

## Done! Now download the files

In Colab's left sidebar, go to **Files** > navigate to `/content/drive/MyDrive/eurosat_model/`.
Download these 2 files:
- `ResNet50_eurosat.h5`
- `class_indices.npy`

Then place them in your local `models/` folder, overwriting the existing ones.

Restart the Streamlit app and it'll use the trained model!